In [4]:
import pandas as pd
import torch
import numpy as np
from models import Autoencoder
from utils import get_device, crear_datasets_proporcionales, estandarizar_columnas

df = pd.read_csv("data/diabetes_012_health_indicators_BRFSS2015.csv")

# "Binarizamos" los datos, eliminando registros de pacientes con prediabetes
df = df[df["Diabetes_012"] != 1]
df["Diabetes_012"] = df["Diabetes_012"].replace(2, 1)

# Estandarizamos las columnas no binarias
df = estandarizar_columnas(df=df, cols_estandarizar=["BMI", "MentHlth", "PhysHlth", "Age", "Education", "Income"])
list_x_train, list_y_train, list_x_test, list_y_test, resumen_df = crear_datasets_proporcionales(df, "Diabetes_012")

device = get_device()
x0, x1, x2, x3 = list_x_test
print(resumen_df)


Dispositivo usado: cuda (NVIDIA GeForce GTX 970)

   Proporción  Positivos  Negativos  Total  % Positivos  % Negativos
0        0.00          0      70692  70692          0.0        100.0
1        0.10       7069      63623  70692         10.0         90.0
2        0.25      17673      53019  70692         25.0         75.0
3        0.50      35346      35346  70692         50.0         50.0


In [6]:
modelo = Autoencoder.load(path="models/autoencoder_00-02_13-11-25_0.pth", device=device)
original = x0[0]

reconstruido = modelo.predict(x=original, device=device)
prediccion = np.round(np.abs(reconstruido))

aciertos = np.sum(original == prediccion)
total = original.size

print(f"--- Comparación de Reconstrucción ---")
print(f"{aciertos}/{total} valores reconstruidos correctamente.")
print(f"Aciertos: {100 * aciertos / total:.2f}%\n")

print("Array original:")
print(original)
print("\nArray reconstruido:")
print(prediccion)


Modelo cargado correctamente de 'models/autoencoder_00-02_13-11-25_0.pth'
--- Comparación de Reconstrucción ---
11/21 valores reconstruidos correctamente.
Aciertos: 52.38%

Array original:
[ 0.          0.          0.         -0.50633874  1.          0.
  0.          1.          0.          0.          0.          0.
  1.          3.         -0.42814476 -0.48414917  0.          0.
 -0.3311125   0.96059071 -2.45224647]

Array reconstruido:
[0. 0. 1. 0. 1. 0. 0. 1. 0. 1. 0. 1. 0. 3. 0. 0. 0. 0. 0. 1. 1.]


In [7]:
from utils import evaluar_anomalias, obtener_metricas

x0, x1, x2, x3 = list_x_test
y0, y1, y2, y3 = list_y_test

resultados = evaluar_anomalias(modelo, x0, y0, device, 1)
print(resultados)

metricas = obtener_metricas(**resultados)

print("Matriz de confusión (valores):")
print(metricas["matriz_confusion"])
print("\nMatriz de confusión (%):")
print(metricas["matriz_confusion_pct"])

print(f"\nAccuracy : {metricas['accuracy']}")
print(f"Precisión: {metricas['precision']}")
print(f"Recall   : {metricas['recall']}")
print(f"F1-Score : {metricas['f1_score']}")

{'TP': 19151, 'FN': 16195, 'TN': 82140, 'FP': 60871}
Matriz de confusión (valores):
[[82140 60871]
 [16195 19151]]

Matriz de confusión (%):
[[46.05 34.13]
 [ 9.08 10.74]]

Accuracy : 0.5679
Precisión: 0.2393
Recall   : 0.5418
F1-Score : 0.332
